# Bài thực hành 03 — Metadata, RBAC & Chatbot

Tái sử dụng index Bài 01, lọc quyền trước retrieval và ghi audit log.

In [5]:
from pathlib import Path
import sys
here=Path.cwd().resolve()
project=next(p for p in (here,*here.parents,here/'RAG_Demo_Labs') if (p/'rag_gemini_runtime.py').exists())
sys.path.insert(0,str(project))
print('Thư mục dự án:',project)
from rag_gemini_runtime import STORE_DIR,load_index,retrieve,infer
import json
from datetime import datetime

Thư mục dự án: C:\Users\HP\Downloads\RAG_Demo_Labs


In [6]:
records=load_index('foundation')
catalog={'TT_02_2023_NHNN.md':{'classification':'public','allowed_roles':['business_user','credit_officer','compliance','internal_auditor']},'TT_06_2023_NHNN.md':{'classification':'internal','allowed_roles':['credit_officer','compliance','internal_auditor']},'TT_39_2016_NHNN.md':{'classification':'confidential','allowed_roles':['credit_officer','compliance','internal_auditor']},'chinh_sach_tin_dung.md':{'classification':'restricted','allowed_roles':['compliance','internal_auditor']}}
clearance={'business_user':{'public'},'credit_officer':{'public','internal','confidential'},'compliance':{'public','internal','confidential','restricted'},'internal_auditor':{'public','internal','confidential','restricted'}}
STORE_DIR.mkdir(exist_ok=True)
(STORE_DIR/'metadata_catalog.json').write_text(json.dumps(catalog,ensure_ascii=False,indent=2),encoding='utf-8')
print('Đã tạo metadata catalog.')

Đã tạo metadata catalog.


In [7]:
def chat(message, history, role):
    # Filter records based on Role-Based Access Control (RBAC) and document classification clearance
    allowed = [
        r for r in records 
        if role in catalog[r['source']]['allowed_roles'] 
        and catalog[r['source']]['classification'] in clearance[role]
    ]
    
    hits = retrieve(message, allowed, top_k=5)
    
    # Pass the role context and compliance instructions to the inference engine
    answer = infer(message, hits, f'Vai trò: {role}. Tuân thủ RBAC và trích dẫn nguồn.')
    
    citations = '\n'.join(f'- [SOURCE: {h["source"]} | chunk {h["chunk_id"]}]' for h in hits)
    return answer + '\n\n' + citations

In [8]:
import gradio as gr
demo=gr.ChatInterface(fn=chat,additional_inputs=gr.Dropdown(choices=list(clearance),value='business_user',label='Vai trò'),title='Chatbot RAG có Metadata & RBAC',description='RBAC lọc tài liệu trước retrieval.')
demo.launch()

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
